# arms_ci95 — selected checkpoint of each training arm against the unadapted model, with 95% intervals

Main test set, 488 rows. Means of PF, SC and Overall for the unadapted model and the selected checkpoint of each
training arm (LoRA r4, r8, r16 at step 1,000; QLoRA r8 at step 2,000; soft prompting (i) and (iii) at step 2,000), and
the paired differences from the unadapted model (ΔSC, ΔOverall) with t-based 95% intervals. Also the parameter-matched
contrast LoRA r4 − soft prompting (iii) on Overall with its 95% interval.

Reads the per-row judge files in `Data/scores/` only. Section 3 checks every mean, difference and standard error
against the values printed by `lora_check`, `qlora_check` and `soft_prompt_check`.

## 1. Inputs

In [ ]:
import json, math
from pathlib import Path
import numpy as np
from scipy import stats

CANDIDATES = [Path.cwd(),
              Path(r"D:\Documents\My Documents\University\MSc in Artificial Intelligence and Data Engineering"
                   r"\Master's Thesis\My Master's Thesis\Master_Thesis_FULL\My Master's Thesis")]
ROOT = next((p for p in CANDIDATES if (p / "Data" / "scores").is_dir()), None)
assert ROOT is not None, "Data/scores not found under the notebook folder or the local thesis root"
SCORES = ROOT / "Data" / "scores"


def load_by_uid(name):
    d = {}
    with open(SCORES / f"{name}.jsonl", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if "error" in r:
                continue
            d[r["uid"]] = r
    return d


def guessed(d):
    return sum(str(r.get("PF_reason", "")).startswith("guess_if_cannot_parse")
               or str(r.get("SC_reason", "")).startswith("guess_if_cannot_parse") for r in d.values())


ov = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))
MET = {"PF": lambda r: r["PF"], "SC": lambda r: r["SC"], "Overall": ov}

RUNS = {  # table label -> score file
    "Unadapted":                  "test__baseline",
    "LoRA r4 (1,000)":            "test__lora_r4_step1000",
    "LoRA r8 (1,000)":            "test__lora_r8_step1000",
    "LoRA r16 (1,000)":           "test__lora_r16_step1000",
    "QLoRA r8 (2,000)":           "test__qlora_r8_step2000",
    "Soft prompting i (2,000)":   "test__sp_i_p668_step2000",
    "Soft prompting iii (2,000)": "test__sp_iii_p139_step2000",
}
D = {lab: load_by_uid(f) for lab, f in RUNS.items()}
I = sorted(set.intersection(*[set(d) for d in D.values()]))
assert len(I) == 488, f"expected 488 paired test rows, found {len(I)}"

print("root:", ROOT)
print(f"test rows paired across the seven runs : {len(I)}")
print(f"guessed judge scores in the seven runs : {sum(guessed(d) for d in D.values())}")

root: d:\Documents\My Documents\University\MSc in Artificial Intelligence and Data Engineering\Master's Thesis\My Master's Thesis\Master_Thesis_FULL\My Master's Thesis
test rows paired across the seven runs : 488
guessed judge scores in the seven runs : 0


## 2. Means and paired differences

Differences are run minus unadapted, paired per row on `uid`. Interval: mean ± t(0.975, n−1)·SE, with SE the standard
deviation of the per-row differences over √n. p from the paired t-test.

In [2]:
def paired(A, B, f):
    a = np.array([f(A[u]) for u in I]); b = np.array([f(B[u]) for u in I])
    d = a - b; n = len(d)
    se = d.std(ddof=1) / math.sqrt(n); half = stats.t.ppf(0.975, n - 1) * se
    return dict(mean=float(d.mean()), se=float(se), half=float(half),
                lo=float(d.mean() - half), hi=float(d.mean() + half), p=float(stats.ttest_rel(a, b)[1]))


BASE = D["Unadapted"]
MEANS = {lab: {m: float(np.mean([f(d[u]) for u in I])) for m, f in MET.items()} for lab, d in D.items()}
DELTA = {(lab, m): paired(d, BASE, MET[m]) for lab, d in D.items() if lab != "Unadapted" for m in ("SC", "Overall")}
MATCHED = paired(D["LoRA r4 (1,000)"], D["Soft prompting iii (2,000)"], ov)   # LoRA r4 minus prefix-tuning (iii)

print(f"{'run':27s} {'':8s} {'mean':>8s} {'SE':>7s} {'95% half':>9s}  {'95% CI':20s} {'p':>9s}")
for (lab, m), s in DELTA.items():
    print(f"{lab:27s} {'d' + m:8s} {s['mean']:+8.4f} {s['se']:7.4f} {s['half']:9.4f}  "
          f"[{s['lo']:+.4f}, {s['hi']:+.4f}] {s['p']:9.3g}")
s = MATCHED
print(f"\nLoRA r4 (1,000) - Soft prompting iii (2,000), Overall: {s['mean']:+.4f}  SE {s['se']:.4f}  "
      f"95% CI [{s['lo']:+.4f}, {s['hi']:+.4f}]  p {s['p']:.3g}")

run                                      mean      SE  95% half  95% CI                       p
LoRA r4 (1,000)             dSC       -0.2336  0.0676    0.1329  [-0.3665, -0.1007]  0.000601
LoRA r4 (1,000)             dOverall  -0.1147  0.0534    0.1049  [-0.2196, -0.0097]    0.0323
LoRA r8 (1,000)             dSC       -0.2418  0.0690    0.1356  [-0.3774, -0.1062]    0.0005
LoRA r8 (1,000)             dOverall  -0.1197  0.0539    0.1060  [-0.2257, -0.0138]    0.0269
LoRA r16 (1,000)            dSC       -0.2234  0.0693    0.1361  [-0.3595, -0.0872]   0.00135
LoRA r16 (1,000)            dOverall  -0.1109  0.0550    0.1081  [-0.2191, -0.0028]    0.0444
QLoRA r8 (2,000)            dSC       -0.1004  0.0765    0.1503  [-0.2507, +0.0499]      0.19
QLoRA r8 (2,000)            dOverall  -0.0799  0.0623    0.1224  [-0.2023, +0.0425]       0.2
Soft prompting i (2,000)    dSC       -1.5102  0.1205    0.2368  [-1.7470, -1.2734]  2.03e-31
Soft prompting i (2,000)    dOverall  -1.3507  0.1007    0

## 3. Parity with the earlier notebooks

Every mean, difference and SE must print identically, at three decimals, to the value printed by the notebook named
beside it: `lora_check` sections 2 and 3, `qlora_check` sections 2 and 3, `soft_prompt_check` sections 2 and 4.

In [3]:
MEAN_PUB = {  # run: ((PF, SC, Overall) as printed, notebook)
    "Unadapted":                  ((8.004, 8.088, 7.963), "lora_check"),
    "LoRA r4 (1,000)":            ((8.027, 7.855, 7.848), "lora_check"),
    "LoRA r8 (1,000)":            ((8.064, 7.846, 7.843), "lora_check"),
    "LoRA r16 (1,000)":           ((8.033, 7.865, 7.852), "lora_check"),
    "QLoRA r8 (2,000)":           ((7.990, 7.988, 7.883), "qlora_check"),
    "Soft prompting i (2,000)":   ((7.123, 6.578, 6.612), "soft_prompt_check"),
    "Soft prompting iii (2,000)": ((7.830, 6.859, 7.137), "soft_prompt_check"),
}
DELTA_PUB = {  # (run, metric): (difference, SE as printed, notebook)
    ("LoRA r4 (1,000)", "SC"):            (-0.234, 0.068, "lora_check"),
    ("LoRA r4 (1,000)", "Overall"):       (-0.115, 0.053, "lora_check"),
    ("LoRA r8 (1,000)", "SC"):            (-0.242, 0.069, "lora_check"),
    ("LoRA r8 (1,000)", "Overall"):       (-0.120, 0.054, "lora_check"),
    ("LoRA r16 (1,000)", "SC"):           (-0.223, 0.069, "lora_check"),
    ("LoRA r16 (1,000)", "Overall"):      (-0.111, 0.055, "lora_check"),
    ("QLoRA r8 (2,000)", "SC"):           (-0.100, 0.076, "qlora_check"),
    ("QLoRA r8 (2,000)", "Overall"):      (-0.080, 0.062, "qlora_check"),
    ("Soft prompting i (2,000)", "SC"):   (-1.510, 0.121, "soft_prompt_check"),
    ("Soft prompting i (2,000)", "Overall"): (-1.351, 0.101, "soft_prompt_check"),
    ("Soft prompting iii (2,000)", "SC"): (-1.230, 0.105, "soft_prompt_check"),
    ("Soft prompting iii (2,000)", "Overall"): (-0.826, 0.083, "soft_prompt_check"),
}
MATCHED_PUB = (-0.711, 0.076, "soft_prompt_check")  # printed there as (iii) - LoRA r4; compared with the sign flipped

same = lambda mine, pub: f"{mine:+.3f}" == f"{pub:+.3f}"
bad = []
for lab, (vals, src) in MEAN_PUB.items():
    for m, v in zip(("PF", "SC", "Overall"), vals):
        if not same(MEANS[lab][m], v): bad.append(f"{lab} {m} mean: here {MEANS[lab][m]:.4f}, {src} {v:.3f}")
for (lab, m), (dv, sv, src) in DELTA_PUB.items():
    s = DELTA[(lab, m)]
    if not same(s["mean"], dv): bad.append(f"{lab} d{m}: here {s['mean']:+.4f}, {src} {dv:+.3f}")
    if not same(s["se"], sv):   bad.append(f"{lab} d{m} SE: here {s['se']:.4f}, {src} {sv:.3f}")
if not same(-MATCHED["mean"], MATCHED_PUB[0]): bad.append(f"matched difference: here {-MATCHED['mean']:+.4f}")
if not same(MATCHED["se"], MATCHED_PUB[1]):    bad.append(f"matched SE: here {MATCHED['se']:.4f}")
n_checked = 3 * len(MEAN_PUB) + 2 * len(DELTA_PUB) + 2
assert not bad, "\n".join(bad)
print(f"all {n_checked} means, differences and SEs reproduce the values printed by lora_check, qlora_check "
      f"and soft_prompt_check")

all 47 means, differences and SEs reproduce the values printed by lora_check, qlora_check and soft_prompt_check


## 4. Summary table

Means to three decimals; differences and 95% half-widths to two decimals.

In [4]:
print(f"{'Method (step)':27s} {'PF':>6s} {'SC':>6s} {'Overall':>7s}   {'dSC':>12s}   {'dOverall':>12s}")
for lab in RUNS:
    m = MEANS[lab]
    row = f"{lab:27s} {m['PF']:6.3f} {m['SC']:6.3f} {m['Overall']:7.3f}"
    if lab != "Unadapted":
        row += "".join(f"   {DELTA[(lab, k)]['mean']:+.2f} ± {DELTA[(lab, k)]['half']:.2f}" for k in ("SC", "Overall"))
    print(row)
s = MATCHED
print(f"\nLoRA r4 (1,000) - Soft prompting iii (2,000), Overall: {s['mean']:.2f} "
      f"(95% CI {s['lo']:.2f} to {s['hi']:.2f}, p {s['p']:.2g})")

Method (step)                   PF     SC Overall            dSC       dOverall
Unadapted                    8.004  8.088   7.963
LoRA r4 (1,000)              8.027  7.855   7.848   -0.23 ± 0.13   -0.11 ± 0.10
LoRA r8 (1,000)              8.064  7.846   7.843   -0.24 ± 0.14   -0.12 ± 0.11
LoRA r16 (1,000)             8.033  7.865   7.852   -0.22 ± 0.14   -0.11 ± 0.11
QLoRA r8 (2,000)             7.990  7.988   7.883   -0.10 ± 0.15   -0.08 ± 0.12
Soft prompting i (2,000)     7.123  6.578   6.612   -1.51 ± 0.24   -1.35 ± 0.20
Soft prompting iii (2,000)   7.830  6.859   7.137   -1.23 ± 0.21   -0.83 ± 0.16

LoRA r4 (1,000) - Soft prompting iii (2,000), Overall: 0.71 (95% CI 0.56 to 0.86, p 3.7e-19)
